# Part 2: Training and evaluating a predictor

This notebook trains a Transformer predictor of LUT usage on the `wa-hls4ml` dataset, then evaluates it on designs like its training data and on hand-built models absent from it.

It runs from a clone of the repository, with the notebook dependencies installed from the repository root:

```bash
uv sync --group notebooks           # with uv
pip install -e . --group notebooks  # with pip
```

## Downloading the dataset

`wa-hls4ml` holds train, validation and test splits, each made of one JSON file per generator family, and a set of exemplar models.

The full dataset is about 5 GB of JSON, and preparing it can take a few minutes and up to about 10 GB of memory. Downloads are cached by `huggingface_hub`.

In [ ]:
from huggingface_hub import hf_hub_download

families = ["2_20", "2layer", "3layer", "conv1d", "conv2d", "latency", "resource"]


def download(path):
    return hf_hub_download(
        "fastmachinelearning/wa-hls4ml",
        path,
        repo_type="dataset",
        revision="bb0e7d5533c79b218faf59afa2798559cb84838e",
    )


json_files = {
    split: [download(f"{split}/{split}_{family}_merged.json") for family in families]
    for split in ("train", "val", "test")
}
json_files["exemplar"] = [download("exemplar/exemplar_models.json")]

## Preparing the records

`dataset.prepare` parses JSON records once into a typed `.npz` file, holding every feature `rule4ml` extracts and the raw targets of each report.

`dataset.load` reads it back as a `Dataset`.

In [ ]:
import os

from rule4ml import dataset

os.makedirs("data", exist_ok=True)
data = {}
for split, paths in json_files.items():
    path = os.path.join("data", f"{split}_{'-'.join(families)}.npz")
    if not os.path.exists(path):
        dataset.prepare(paths, path)
    data[split] = dataset.load(path)
    print(f"{split}: {len(data[split])} records")

## Choosing features and targets

A predictor reads global features of the whole model and per-layer features, by name. Here we use the same features as the default LUT predictor, read from its config file.

The target is `lut`, read from `resource_report`, the Vivado logic synthesis report (For the HLS estimates instead, read from `hls_resource_report`)

In [ ]:
import json

import rule4ml

weights_dir = os.path.join(
    os.path.dirname(rule4ml.__file__), "models", "weights", "default"
)
with open(os.path.join(weights_dir, "LUT.config.json")) as config_file:
    default_config = json.load(config_file)

global_labels = default_config["global_numerical_labels"]
layer_labels = default_config["sequential_numerical_labels"]
target_labels = ["lut"]
print(f"{len(global_labels)} global features: {global_labels}")
print(f"{len(layer_labels)} layer features: {layer_labels}")

## Building the model

We will build and train a smaller LUT surrogate based on the Transformer architecture. `TorchTransformer` reads the layers of a model as a sequence.

Categorical inputs, such as the strategy or the layer type go through ordinal mapping first, then embedding layers.

Targets are trained on a log scale with `Log1pScaler`, and numerical inputs are compressed with `AsinhInputScaler`, same as the default predictors.

In [ ]:
from rule4ml.models.architectures import TorchTransformer, TransformerSettings
from rule4ml.models.scaling import AsinhInputScaler, Log1pScaler
from rule4ml.parsers.data_parser import (
    default_board_map,
    default_hls4ml_map,
    default_layer_type_map,
    default_strategy_map,
    default_vivado_map,
)
from rule4ml.surrogates import TorchSurrogate

global_maps = {
    "strategy": default_strategy_map,
    "board": default_board_map,
    "hls4ml_version": default_hls4ml_map,
    "vivado_version": default_vivado_map,
}
layer_maps = {"layer_type": default_layer_type_map}

model = TorchTransformer(
    TransformerSettings(d_model=64, num_heads=4, num_layers=2, feedforward_dim=128),
    global_input_shape=(None, len(global_labels) + len(global_maps)),
    sequential_input_shape=(None, len(layer_labels) + len(layer_maps)),
    output_shape=(None, len(target_labels)),
    global_categorical_maps=global_maps,
    sequential_categorical_maps=layer_maps,
    name="LUT_TUTORIAL",
)

predictor = TorchSurrogate()
predictor.set_model(
    model,
    global_labels,
    target_labels,
    layer_labels,
    report="resource_report",
    scaler=Log1pScaler(),
    input_scaler=AsinhInputScaler(),
)

## Training

Training on the full dataset can take some time.

In [ ]:
from rule4ml.models.callbacks import EarlyStopping
from rule4ml.surrogates import TrainSettings

history = predictor.fit(
    data["train"],
    TrainSettings(
        num_epochs=20, batch_size=256, learning_rate=1e-3, loss_function="mse"
    ),
    val_dataset=data["val"],
    callbacks=[EarlyStopping(monitor="val_loss", patience=3)],
    verbose=1,
)

## Evaluating

Calling `evaluate` reports SMAPE, RMSE and R² per target, over the records where the target is known.

- **Test split:** designs from the same data generators, kept out of training (in distribution).
- **Exemplar models:** seven hand-built model families absent from training (out of distribution).

The default LUT predictor, trained on the full dataset, gives a reference on the same records.

In [ ]:
import pandas as pd

default_lut = TorchSurrogate()
default_lut.load(
    os.path.join(weights_dir, "LUT.config.json"),
    os.path.join(weights_dir, "LUT.weights.pt"),
)

scores = {
    (name, split): surrogate.evaluate(records)["lut"]
    for name, surrogate in (("tutorial", predictor), ("default", default_lut))
    for split, records in (("test", data["test"]), ("exemplar", data["exemplar"]))
}
pd.DataFrame(scores).T

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (split, records) in zip(
    axes, (("test", data["test"]), ("exemplar", data["exemplar"]))
):
    true = records.targets(target_labels, "resource_report")[:, 0]
    known = np.isfinite(true)
    for name, surrogate in (("tutorial", predictor), ("default", default_lut)):
        predicted = surrogate.predict_dataset(records)[:, 0]
        ax.scatter(true[known], predicted[known], s=4, alpha=0.3, label=name)

    limits = [1, max(true[known].max(), 10)]
    ax.plot(limits, limits, "k--", linewidth=1)
    ax.set(
        xscale="log",
        yscale="log",
        title=split,
        xlabel="Synthesized LUT",
        ylabel="Predicted LUT",
    )
    ax.legend()

plt.show()

## Saving and loading

`save` writes the config as JSON and the weights. `load` restores the predictor.

In [ ]:
predictor.save("models")

reloaded = TorchSurrogate()
reloaded.load("models/LUT_TUTORIAL.config.json", "models/LUT_TUTORIAL.weights.pt")
np.testing.assert_allclose(
    reloaded.predict_dataset(data["exemplar"]),
    predictor.predict_dataset(data["exemplar"]),
)

## Other architectures

`TorchGNN` (with `GNNSettings`) reads the layers as a graph, and `KerasMLP` (with `MLPSettings`) reads global features only.

Both plug into the same `set_model`, `fit`, `evaluate`, `save` and `load` calls, with `KerasSurrogate` (needs `rule4ml[keras]`) for the KerasMLP.